eFL-Boost with the per-class Builder + multi-objective optimisation

Based on eFL-Boost: Efficient Federated Learning for GBDT (Yamamoto, Ozawa & Wang, IEEE Access 2022)

Horizontal federated GBDT in which **the tree structure is decided locally by one client (the Builder)** and **the leaf
weights are computed globally** from the gradients of all clients. Each boosting round needs three communications:

1. **Builder → clients:** the Builder grows the trees on its own data and sends their structure (splits only, no leaf values).
2. **Clients → aggregator:** every client routes its own samples through the trees and sends the sum of gradients and
   Hessians in each leaf.
3. **Aggregator → clients:** leaf values `w = −ΣG / (ΣH + λ) · η` are computed from the summed statistics and sent back to everyone.

Multi-class uses softmax, with one tree per class per round, as in FedTree.ipynb. The histogram, split and tree code is
reused unchanged from FedTree.ipynb.

### Notes on the paper and the implementation choices

* **This notebook changes the Builder rule; everything else is eFL-Boost as in the paper.** The paper's aggregator
  selects the Builder "based on a predetermined order" (`builder_rule="round_robin"`, client `t mod M` builds every
  class tree of round `t`) and names a better Builder criterion as future work. Here the default is
  **`builder_rule="per_class"`**: for each class `k`, the Builder of the class-`k` tree is drawn at random with
  probability proportional to how many class-`k` samples each client holds, so the class-`k` tree is grown by a client
  that actually has class `k` (with non-IID labels, most clients hold few or none of the rare classes). Only the choice
  of Builder changes: gradients, local structure, per-leaf G/H sums and global leaf weights are unchanged.
* **Cost of the per-class Builder:** the aggregator needs each client's class counts, sent **once before training**:
  one extra communication (151 instead of 150 for 50 rounds, `setup_mb`), and it reveals each client's label
  distribution to the aggregator. Each round is still 3 communications: the K class trees of a round (now from up to
  K different Builders) travel together.
* **Compare with `eFLBoost_1.ipynb`** (paper's round-robin Builder, same multi-objective search): same data split,
  same search space, same selection rule; only the Builder rule differs. `builder_rule="round_robin"` in `moo_tune`
  and `run_eflboost` reproduces that notebook.
* **Hyperparameters are not part of the algorithm.** η, λ, the number of bins, the depth and the number of rounds can be
  changed without changing eFL-Boost; the multi-objective search below chooses them.
* **Cut points.** The Builder uses quantile cut points of **its own** data only (`CutPoints.compute` on the Builder's
  data, cached per client). No feature information is shared.
* **Losses and hyperparameters** are FedTree's: softmax gradient `p − y`, Hessian `p(1 − p)`, λ = 0.1, γ = 0.001,
  min_child_weight = 0, 64 bins, depth 6, η = 0.1, 50 rounds, seed 42.
* **Communication size** is measured as in FedTree, with `sizeof_mb` (pickled payload size), for every message actually sent.
* **Multi-objective optimisation (maximise macro-F1, minimise testing time).** `moo_tune` searches `n_trees` 10–100, `max_depth` 2–8,
  `learning_rate` 0.05–0.5, `n_bins` 32/64/128 and `lam` 0.01–10 with **NSGA-II** (Optuna's `NSGAIISampler`), maximising
  macro-F1 and minimising the testing time per sample, both measured on a validation split of the training data (the
  test set is never seen). Each trial trains one model and also scores its first 10, 20, … rounds (training for fewer
  rounds gives exactly that prefix, since the Builder draws of round `t` do not
  depend on later rounds). From the Pareto front it returns the fastest setting within `f1_tolerance` (1.0)
  macro-F1 points of the best, or with `f1_tolerance=None` the knee point. A **convergence chart** rebuilds the
  front after every 5 trials and plots its hypervolume (the area of the F1/time plane it dominates), the best
  F1 and fastest time so far, and the point that would be selected; the first checkpoint at 99 % of the final
  hypervolume tells how many trials are enough (`MOO_FRONTS[(dataset, partition)]["convergence"]`). The tuned run is added next to the original
  run of each dataset and partition, so the summary table compares both.

# Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional, Any
from collections import defaultdict, Counter
import time
import sys
import pickle
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import LabelEncoder
import copy
import io
import contextlib

# Utility funcs (from FedTree.ipynb)

In [ ]:
def sizeof_mb(obj: Any) -> float:
    return sys.getsizeof(pickle.dumps(obj, protocol=pickle.HIGHEST_PROTOCOL)) / (1024 * 1024)

def measure_inference_time(predict_fn, X, n_runs: int = 5) -> dict:
    """Measure total inference time and time-per-sample (median of several runs)."""
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        _ = predict_fn(X)
        times.append(time.perf_counter() - t0)
    total = float(np.median(times))
    per_sample = total / max(len(X), 1) * 1000  # milliseconds
    return {
        "test_time_total_s": total,
        "test_time_per_sample_ms": per_sample,
    }


def softmax(scores: np.ndarray) -> np.ndarray:
    scores = scores - scores.max(axis=1, keepdims=True)
    exp = np.exp(scores)
    return exp / exp.sum(axis=1, keepdims=True)

# Cut points (from FedTree.ipynb)

In [ ]:
@dataclass
class CutPoints:
    boundaries: Dict[int, np.ndarray]   # feature -> sorted cut points

    @staticmethod
    def compute(X: np.ndarray, n_bins: int = 32) -> "CutPoints":
        boundaries = {}
        quantiles = np.linspace(0, 1, n_bins + 1)[1:-1]
        for f in range(X.shape[1]):
            col = X[:, f]
            cuts = np.unique(np.quantile(col, quantiles))
            if len(cuts) == 0:
                cuts = np.array([np.median(col)])
            boundaries[f] = cuts
        return CutPoints(boundaries=boundaries)

# Histogram helpers (from FedTree.ipynb)

In [ ]:
def compute_local_histogram(
    X: np.ndarray, g: np.ndarray, h: np.ndarray,
    instance_ids: np.ndarray, cuts: CutPoints
) -> Dict[int, np.ndarray]:
    """Returns {feat: array of shape (n_bins+1, 2)}  [sum_g, sum_h]"""
    hist = {}
    if len(instance_ids) == 0:
        for f, bnds in cuts.boundaries.items():
            hist[f] = np.zeros((len(bnds) + 1, 2), dtype=np.float64)
        return hist

    Xn = X[instance_ids]
    gn = g[instance_ids]
    hn = h[instance_ids]

    for f, bnds in cuts.boundaries.items():
        bin_idx = np.searchsorted(bnds, Xn[:, f], side="left")   # bin i  <=>  x <= bnds[i]
        n_bins = len(bnds) + 1
        gh = np.zeros((n_bins, 2), dtype=np.float64)
        np.add.at(gh[:, 0], bin_idx, gn)
        np.add.at(gh[:, 1], bin_idx, hn)
        hist[f] = gh
    return hist


def aggregate_histograms(client_hists: List[Dict[int, np.ndarray]]) -> Dict[int, np.ndarray]:
    agg = {f: np.zeros_like(h) for f, h in client_hists[0].items()}
    for hist in client_hists:
        for f, gh in hist.items():
            agg[f] += gh
    return agg

# Split gain & leaf value (from FedTree.ipynb)

In [ ]:
def leaf_value(G: float, H: float, lam: float) -> float:
    return -G / (H + lam)


def split_gain(GL, HL, GR, HR, lam, gamma) -> float:
    def term(G, H):
        return (G * G) / (H + lam) if (H + lam) > 1e-12 else 0.0
    return 0.5 * (term(GL, HL) + term(GR, HR) - term(GL + GR, HL + HR)) - gamma


def find_best_split(agg_hist: Dict[int, np.ndarray], cuts: CutPoints,
                    lam: float, gamma: float, min_child_weight: float = 1.0
                    ) -> Optional[Tuple[int, float, float]]:
    best_gain, best_feat, best_cut = -np.inf, None, None

    for f, gh in agg_hist.items():
        bnds = cuts.boundaries[f]
        cum_g = np.cumsum(gh[:, 0])
        cum_h = np.cumsum(gh[:, 1])
        G_tot, H_tot = cum_g[-1], cum_h[-1]

        for i in range(len(bnds)):
            GL, HL = cum_g[i], cum_h[i]
            GR, HR = G_tot - GL, H_tot - HL
            if HL < min_child_weight or HR < min_child_weight:
                continue
            gain = split_gain(GL, HL, GR, HR, lam, gamma)
            if gain > best_gain:
                best_gain = gain
                best_feat = f
                best_cut = bnds[i]

    if best_feat is None or best_gain <= 0:
        return None
    return best_feat, best_cut, best_gain

# Tree node & prediction (from FedTree.ipynb)

In [ ]:
@dataclass
class TreeNode:
    is_leaf: bool = False
    feature: Optional[int] = None
    threshold: Optional[float] = None
    value: float = 0.0
    left: Optional["TreeNode"] = None
    right: Optional["TreeNode"] = None


def predict_tree(node: TreeNode, X: np.ndarray) -> np.ndarray:
    out = np.zeros(X.shape[0])
    def _rec(n, idx):
        if n.is_leaf or len(idx) == 0:
            out[idx] = n.value
            return
        go_left = X[idx, n.feature] <= n.threshold
        _rec(n.left, idx[go_left])
        _rec(n.right, idx[~go_left])
    _rec(node, np.arange(X.shape[0]))
    return out

# Tree building (from FedTree.ipynb)

eFL-Boost's Builder calls this with a **single client** (itself), so the structure comes from its own data only.
The leaf values it computes locally are discarded and replaced by the global ones.

In [ ]:
def build_one_federated_tree(
    client_X: Dict[int, np.ndarray],
    client_g: Dict[int, np.ndarray],
    client_h: Dict[int, np.ndarray],
    cuts: CutPoints,
    max_depth: int,
    lam: float,
    gamma: float,
    learning_rate: float,
    min_child_weight: float,
    comm_tracker: Dict[str, float],
) -> TreeNode:

    client_ids = list(client_X.keys())
    client_inst = {cid: np.arange(client_X[cid].shape[0]) for cid in client_ids}

    root = TreeNode()
    frontier = [(root, client_inst, 0)]   # (node, {cid: ids}, depth)

    while frontier:
        node, inst_map, depth = frontier.pop(0)

        # --- PartyComputeHistogram ---
        client_hists = []
        for cid in client_ids:
            hist = compute_local_histogram(
                client_X[cid], client_g[cid], client_h[cid],
                inst_map[cid], cuts
            )
            client_hists.append(hist)
            comm_tracker["upload_mb"] += sizeof_mb(hist)

        # --- ServerAggregate ---
        agg = aggregate_histograms(client_hists)

        # Correct total G / H (any feature is fine)
        any_f = next(iter(agg))
        G_total = agg[any_f][:, 0].sum()
        H_total = agg[any_f][:, 1].sum()

        can_split = depth < max_depth
        split = find_best_split(agg, cuts, lam, gamma, min_child_weight) if can_split else None

        if split is None:
            node.is_leaf = True
            node.value = leaf_value(G_total, H_total, lam) * learning_rate
            continue

        feat, thresh, _ = split
        node.feature = feat
        node.threshold = thresh
        node.left = TreeNode()
        node.right = TreeNode()
        comm_tracker["download_mb"] += sizeof_mb((feat, thresh))

        # --- PartyUpdate: split instance sets ---
        next_left, next_right = {}, {}
        for cid in client_ids:
            ids = inst_map[cid]
            if len(ids) == 0:
                next_left[cid] = ids
                next_right[cid] = ids
                continue
            vals = client_X[cid][ids, feat]
            mask = vals <= thresh
            next_left[cid] = ids[mask]
            next_right[cid] = ids[~mask]

        frontier.append((node.left, next_left, depth + 1))
        frontier.append((node.right, next_right, depth + 1))

    return root

# Partitioning (from FedTree.ipynb)

In [ ]:
def partition_iid(X: np.ndarray, y: np.ndarray, n_clients: int, seed: int = 42):
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(y))
    splits = np.array_split(idx, n_clients)
    return {i: (X[s], y[s]) for i, s in enumerate(splits)}


def partition_dirichlet(X: np.ndarray, y: np.ndarray, n_clients: int,
                        alpha: float = 0.5, seed: int = 42):
    rng = np.random.default_rng(seed)
    labels = np.unique(y)
    client_idx = defaultdict(list)

    for c in labels:
        idx_c = np.where(y == c)[0]
        rng.shuffle(idx_c)
        proportions = rng.dirichlet([alpha] * n_clients)
        proportions = (np.cumsum(proportions) * len(idx_c)).astype(int)[:-1]
        splits = np.split(idx_c, proportions)
        for i, s in enumerate(splits):
            client_idx[i].extend(s)

    return {i: (X[idxs], y[idxs]) for i, idxs in client_idx.items()}

# eFL-Boost helpers

* **Structure only:** a copy of the tree with every leaf value zeroed, which is what the Builder sends.
* **Routing:** each client finds the leaf index of each of its samples. Leaves are numbered in the same depth-first order everywhere.
* **Builder rule:** chooses which client builds each class tree of a round: `round_robin` (paper, one Builder per
  round), `random`, or `per_class` (a client drawn in proportion to its samples of that class).

In [ ]:
def collect_leaves(node: TreeNode) -> List[TreeNode]:
    """Leaves in depth-first (left-to-right) order; this order indexes the leaf statistics."""
    if node.is_leaf:
        return [node]
    return collect_leaves(node.left) + collect_leaves(node.right)


def count_nodes(node: TreeNode) -> int:
    return 1 if node.is_leaf else 1 + count_nodes(node.left) + count_nodes(node.right)


def structure_only(node: TreeNode) -> TreeNode:
    """Copy of the tree with leaf values removed: what the Builder shares."""
    tree = copy.deepcopy(node)
    for leaf in collect_leaves(tree):
        leaf.value = 0.0
    return tree


def route_to_leaves(node: TreeNode, X: np.ndarray) -> np.ndarray:
    """Leaf index (in collect_leaves order) of every row of X."""
    out = np.zeros(X.shape[0], dtype=np.int64)
    counter = [0]
    def _rec(n, idx):
        if n.is_leaf:
            out[idx] = counter[0]
            counter[0] += 1
            return
        go_left = X[idx, n.feature] <= n.threshold
        _rec(n.left, idx[go_left])
        _rec(n.right, idx[~go_left])
    _rec(node, np.arange(X.shape[0]))
    return out


def choose_builders(t: int, client_ids: List[int], sizes: Dict[int, int], class_counts: Dict[int, np.ndarray],
                    K: int, rule: str, rng: np.random.Generator, min_samples: int) -> List[int]:
    """Builder of each of the K class trees of boosting round t."""
    eligible = [c for c in client_ids if sizes[c] >= min_samples]
    if rule == "round_robin":
        return [eligible[t % len(eligible)]] * K
    if rule == "random":
        return [int(rng.choice(eligible))] * K
    if rule == "per_class":
        builders = []
        for k in range(K):
            w = np.array([class_counts[c][k] for c in eligible], dtype=float)
            if w.sum() == 0:                       # no eligible client holds class k
                builders.append(eligible[t % len(eligible)])
            else:
                builders.append(int(rng.choice(eligible, p=w / w.sum())))
        return builders
    raise ValueError(rule)

# eFL-Boost

`fit` simulates all clients in one process. Each client only touches its own `X`, `y` and scores. Everything that crosses
between clients goes through `_send`, which counts the message and its size. `comm_rounds` is incremented once per
communication step, as it happens.

In [ ]:
@dataclass
class EFLBoostConfig:
    n_trees: int = 50                 # boosting rounds
    max_depth: int = 6
    n_bins: int = 64
    learning_rate: float = 0.1
    lam: float = 0.1
    gamma: float = 0.001
    min_child_weight: float = 0
    random_state: int = 42
    builder_rule: str = "per_class"     # "per_class" | "round_robin" (paper) | "random"
    min_builder_samples: int = 2


class EFLBoost:
    def __init__(self, cfg: EFLBoostConfig):
        self.cfg = cfg
        self.trees: List[List[TreeNode]] = []   # [boosting_round][class]
        self.classes_: np.ndarray = None
        self.builders: List[List[int]] = []     # [boosting_round][class]
        self.comm_rounds = 0
        self.comm_mb = {"setup_mb": 0.0, "structure_mb": 0.0, "leaf_stats_mb": 0.0, "leaf_values_mb": 0.0}
        self.n_messages = 0

    def _send(self, payload, kind: str, n_receivers: int = 1):
        self.comm_mb[kind] += sizeof_mb(payload) * n_receivers
        self.n_messages += n_receivers
        return payload

    def fit(self, client_data: Dict[int, Tuple[np.ndarray, np.ndarray]]):
        cfg = self.cfg
        rng = np.random.default_rng(cfg.random_state)
        client_ids = sorted(client_data.keys())
        client_X = {cid: client_data[cid][0] for cid in client_ids}
        client_y = {cid: client_data[cid][1] for cid in client_ids}
        sizes = {cid: len(client_y[cid]) for cid in client_ids}
        M = len(client_ids)

        self.classes_ = np.unique(np.concatenate([client_y[c] for c in client_ids]))
        K = len(self.classes_)
        class_to_idx = {c: i for i, c in enumerate(self.classes_)}
        client_yi = {cid: np.array([class_to_idx[v] for v in client_y[cid]], dtype=np.int64) for cid in client_ids}
        client_score = {cid: np.zeros((sizes[cid], K)) for cid in client_ids}
        local_cuts = {}                     # each client's own cut points, computed when it first builds

        # ---- setup (per_class only): every client -> aggregator (its class counts), once ----
        class_counts = {cid: np.bincount(client_yi[cid], minlength=K) for cid in client_ids}
        if cfg.builder_rule == "per_class":
            self.comm_rounds += 1
            for cid in client_ids:
                self._send(class_counts[cid], "setup_mb")

        for t in range(cfg.n_trees):
            # ---- local: every client computes softmax gradients of its own samples (global model) ----
            client_g, client_h = {}, {}
            for cid in client_ids:
                p = softmax(client_score[cid])
                g = p.copy()
                g[np.arange(sizes[cid]), client_yi[cid]] -= 1.0
                client_g[cid], client_h[cid] = g, p * (1.0 - p)

            # ---- local at the Builder(s): grow the K tree structures, each on its Builder's own data ----
            builders = choose_builders(t, client_ids, sizes, class_counts, K, cfg.builder_rule, rng,
                                       cfg.min_builder_samples)
            self.builders.append(builders)
            structures = []
            for k, b in enumerate(builders):
                if b not in local_cuts:
                    local_cuts[b] = CutPoints.compute(client_X[b], n_bins=cfg.n_bins)
                local_tree = build_one_federated_tree(
                    {b: client_X[b]}, {b: client_g[b][:, k]}, {b: client_h[b][:, k]}, local_cuts[b],
                    max_depth=cfg.max_depth, lam=cfg.lam, gamma=cfg.gamma, learning_rate=cfg.learning_rate,
                    min_child_weight=cfg.min_child_weight,
                    comm_tracker={"upload_mb": 0.0, "download_mb": 0.0},   # single client: nothing is sent
                )
                structures.append(structure_only(local_tree))

            # ---- communication 1: Builder -> other clients (tree structures) ----
            self.comm_rounds += 1
            self._send(structures, "structure_mb", n_receivers=M - 1)

            # ---- communication 2: every client -> aggregator (per-leaf sums of g and h) ----
            self.comm_rounds += 1
            n_leaves = [len(collect_leaves(s)) for s in structures]
            total = [np.zeros((n_leaves[k], 2)) for k in range(K)]
            client_leaf_idx = {}
            for cid in client_ids:
                stats, idx_k = [], []
                for k in range(K):
                    leaf_idx = route_to_leaves(structures[k], client_X[cid])
                    idx_k.append(leaf_idx)
                    G = np.bincount(leaf_idx, weights=client_g[cid][:, k], minlength=n_leaves[k])
                    H = np.bincount(leaf_idx, weights=client_h[cid][:, k], minlength=n_leaves[k])
                    stats.append(np.stack([G, H], axis=1))
                client_leaf_idx[cid] = idx_k
                for k, s in enumerate(self._send(stats, "leaf_stats_mb")):
                    total[k] += s

            # ---- global at the aggregator: leaf weights from the summed statistics ----
            leaf_values = [np.array([leaf_value(G, H, cfg.lam) * cfg.learning_rate for G, H in total[k]])
                           for k in range(K)]

            # ---- communication 3: aggregator -> all clients (leaf values) ----
            self.comm_rounds += 1
            self._send(leaf_values, "leaf_values_mb", n_receivers=M)
            round_trees = []
            for k in range(K):
                tree = copy.deepcopy(structures[k])
                for leaf, v in zip(collect_leaves(tree), leaf_values[k]):
                    leaf.value = float(v)
                round_trees.append(tree)
                for cid in client_ids:                       # clients update their own scores
                    client_score[cid][:, k] += leaf_values[k][client_leaf_idx[cid][k]]
            self.trees.append(round_trees)

            if (t + 1) % 10 == 0:
                print(f"  Boosting round {t + 1}/{cfg.n_trees} done (builders = {builders})")
        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        scores = np.zeros((X.shape[0], len(self.classes_)))
        for round_trees in self.trees:
            for k, tree in enumerate(round_trees):
                scores[:, k] += predict_tree(tree, X)
        return self.classes_[np.argmax(scores, axis=1)]

    def score(self, X: np.ndarray, y: np.ndarray) -> Dict[str, float]:
        y_pred = self.predict(X)
        return {
            "accuracy": accuracy_score(y, y_pred) * 100,
            "f1_macro": f1_score(y, y_pred, average="macro", zero_division=0) * 100,
            "f1_weighted": f1_score(y, y_pred, average="weighted", zero_division=0) * 100,
        }

# Full pipeline

In [ ]:
ALL_RESULTS = []      # one summary row per run (saved to eflboost_perclass_results.csv at the end)


def run_eflboost(
    df: pd.DataFrame,
    dataset_name: str,
    label_col: str = "Label",
    partition_mode: str = "iid",          # "iid" | "dirichlet"
    n_clients: int = 10,
    dirichlet_alpha: float = 0.5,
    n_trees: int = 50,
    max_depth: int = 6,
    n_bins: int = 64,
    learning_rate: float = 0.1,
    lam: float = 0.1,
    builder_rule: str = "per_class",
    tuned: Any = False,                   # "moo" when the parameters come from moo_tune
    test_size: float = 0.20,
    random_state: int = 42,
    verbose: bool = True,
) -> Dict:

    feature_cols = [c for c in df.columns if c != label_col]
    X = df[feature_cols].values.astype(np.float64)
    y = df[label_col].values

    # Global train/test
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )

    # Partition
    if partition_mode == "iid":
        client_data = partition_iid(X_tr, y_tr, n_clients, seed=random_state)
    else:
        client_data = partition_dirichlet(X_tr, y_tr, n_clients, alpha=dirichlet_alpha, seed=random_state)

    if verbose:
        print(f"Dataset: {dataset_name} {df.shape} | mode={partition_mode} | clients={n_clients}")
        print(f"Params : n_trees={n_trees}, max_depth={max_depth}, learning_rate={learning_rate}, n_bins={n_bins}, "
              f"lam={lam}, builder_rule={builder_rule}, tuned={tuned}")
        for cid, (Xc, yc) in client_data.items():
            print(f"  Client {cid}: {len(yc)} samples, classes={dict(Counter(yc))}")

    cfg = EFLBoostConfig(n_trees=n_trees, max_depth=max_depth, n_bins=n_bins, learning_rate=learning_rate,
                         lam=lam, random_state=random_state, builder_rule=builder_rule)

    # ----- Federated training -----
    t0 = time.perf_counter()
    model = EFLBoost(cfg)
    model.fit(client_data)
    train_time = time.perf_counter() - t0

    metrics = model.score(X_te, y_te)
    inf_stats = measure_inference_time(model.predict, X_te)
    model_size_mb = sizeof_mb(model.trees)
    n_tree_objs = sum(len(r) for r in model.trees)
    n_nodes = sum(count_nodes(t) for r in model.trees for t in r)
    total_comm_mb = sum(model.comm_mb.values())
    y_pred = model.predict(X_te)

    results = {
        "metrics": metrics,
        "efficiency": {
            "n_clients": n_clients,
            "n_boosting_rounds": n_trees,
            "max_depth": max_depth,
            "train_time_s": train_time,
            "test_time_total_s": inf_stats["test_time_total_s"],
            "test_time_per_sample_ms": inf_stats["test_time_per_sample_ms"],
            "model_size_mb": model_size_mb,
            "n_trees": n_tree_objs,
            "n_nodes": n_nodes,
            "n_comm_rounds": model.comm_rounds,
            "n_messages": model.n_messages,
            **model.comm_mb,
            "total_comm_mb": total_comm_mb,
        },
        "builders": model.builders,
        "model": model,
        "y_test": y_te,
        "y_pred": y_pred,
        "classification_report": classification_report(y_te, y_pred, zero_division=0, output_dict=True),
    }
    ALL_RESULTS.append({"dataset": dataset_name, "partition": partition_mode,
                        "alpha": dirichlet_alpha if partition_mode == "dirichlet" else None,
                        "tuned": tuned, "builder_rule": builder_rule, "learning_rate": learning_rate, "n_bins": n_bins, "lam": lam,
                        **metrics, **results["efficiency"]})

    if verbose:
        e = results["efficiency"]
        print("\n===== RESULTS =====")
        print(f"Accuracy     : {metrics['accuracy']:.2f}%")
        print(f"F1-macro     : {metrics['f1_macro']:.2f}%")
        print(f"F1-weighted  : {metrics['f1_weighted']:.2f}%")
        print("\n===== EFFICIENCY =====")
        print(f"Train time          : {train_time:.1f} s")
        print(f"Test time (total)   : {e['test_time_total_s']:.4f} s")
        print(f"Test time / sample  : {e['test_time_per_sample_ms']:.4f} ms")
        print(f"Model size          : {model_size_mb:.3f} MB | {n_tree_objs} trees | {n_nodes} nodes")
        print(f"Total communication : {total_comm_mb:.2f} MB  (setup {e['setup_mb']:.4f} | structures {e['structure_mb']:.2f} | "
              f"leaf stats {e['leaf_stats_mb']:.2f} | leaf values {e['leaf_values_mb']:.2f})")
        print(f"Comm rounds         : {model.comm_rounds}  ({model.n_messages} messages)")
        print(f"Trees built per client: {dict(sorted(Counter(b for r in model.builders for b in r).items()))}")
        print("\nClassification Report:")
        print(classification_report(y_te, y_pred, zero_division=0))

    return results

# Multi-objective optimisation (maximise macro-F1, minimise testing time)

See the notes at the top. One search trains `n_trials` models (each up to 100 rounds): about 20 min per dataset and partition on a laptop. Lower `n_trials` for a quicker, coarser front.

In [ ]:
try:
    import optuna
except ImportError:                # not preinstalled on Colab
    %pip install -q optuna
    import optuna
import matplotlib.pyplot as plt

MOO_SPACE = {
    "n_trees": (10, 100, 10),               # boosting rounds: low, high, step
    "max_depth": (2, 8),
    "learning_rate": (0.05, 0.5),           # log scale
    "n_bins": [32, 64, 128],
    "lam": (0.01, 10.0),                    # log scale
}
MOO_FRONTS = {}       # (dataset, partition) -> {"points": all scored points, "front": Pareto front, "chosen": row}


def pareto_mask(f1: np.ndarray, time_ms: np.ndarray) -> np.ndarray:
    """True for every point that no other point beats on both objectives (higher F1, lower time)."""
    f1, t = np.asarray(f1), np.asarray(time_ms)
    no_worse = (f1[None, :] >= f1[:, None]) & (t[None, :] <= t[:, None])     # [i, j]: j is at least as good as i
    better = (f1[None, :] > f1[:, None]) | (t[None, :] < t[:, None])
    return ~(no_worse & better).any(axis=1)


def knee_point(front: pd.DataFrame) -> int:
    """Front point closest to the ideal corner (best F1, lowest time), both objectives scaled to [0, 1]."""
    f, t = front["val_f1_macro"].values, front["val_time_per_sample_ms"].values
    f_gap = (f.max() - f) / (f.max() - f.min() + 1e-12)
    t_gap = (t - t.min()) / (t.max() - t.min() + 1e-12)
    return int(np.argmin(np.hypot(f_gap, t_gap)))


def pareto_front(points: pd.DataFrame) -> pd.DataFrame:
    front = points[pareto_mask(points["val_f1_macro"], points["val_time_per_sample_ms"])]
    return front.sort_values("val_time_per_sample_ms").reset_index(drop=True)


def select_point(front: pd.DataFrame, f1_tolerance: Optional[float]) -> Tuple[pd.Series, str]:
    """The fastest front point within f1_tolerance macro-F1 points of the best, or the knee point if None."""
    if f1_tolerance is None:
        return front.loc[knee_point(front)], "knee point"
    ok = front[front["val_f1_macro"] >= front["val_f1_macro"].max() - f1_tolerance]
    return ok.loc[ok["val_time_per_sample_ms"].idxmin()], f"fastest within {f1_tolerance} F1 points of the best"


def hypervolume(front: pd.DataFrame, f1_ref: float, t_ref: float) -> float:
    """Area of the (testing time, macro-F1) plane dominated by the front, up to the reference point (f1_ref, t_ref).
    It grows whenever the front improves in either objective. `front` is sorted by time (so F1 increases too)."""
    f, t = front["val_f1_macro"].values, front["val_time_per_sample_ms"].values
    return float(np.sum((np.append(t[1:], t_ref) - t) * (f - f1_ref)))


def convergence_table(points: pd.DataFrame, f1_tolerance: Optional[float], every: int = 5) -> pd.DataFrame:
    """Pareto front of the first 5, 10, 15, ... trials: its hypervolume, the best F1 and fastest time found so far,
    and the point that would be selected. The reference point (worst F1, slowest time of the whole search) is the same
    for every row, so the hypervolumes can be compared."""
    f1_ref, t_ref = points["val_f1_macro"].min(), points["val_time_per_sample_ms"].max()
    n = int(points["trial"].max()) + 1
    rows = []
    for k in sorted(set(range(every, n + 1, every)) | {n}):
        seen = points[points["trial"] < k]
        front = pareto_front(seen)
        chosen, _ = select_point(front, f1_tolerance)
        rows.append({"trials": k, "front_size": len(front),
                     "hypervolume": hypervolume(front, f1_ref, t_ref),
                     "best_f1": seen["val_f1_macro"].max(),
                     "fastest_ms": seen["val_time_per_sample_ms"].min(),
                     "selected_f1": chosen["val_f1_macro"],
                     "selected_ms": chosen["val_time_per_sample_ms"]})
    conv = pd.DataFrame(rows)
    conv["hypervolume_pct"] = 100 * conv["hypervolume"] / conv["hypervolume"].iloc[-1]
    return conv


def plot_convergence(conv: pd.DataFrame, title: str, threshold: float = 99.0):
    """Hypervolume, F1 and testing time of the front after every checkpoint."""
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    ax = axes[0]
    ax.plot(conv["trials"], conv["hypervolume_pct"], "o-", color="C0")
    ax.axhline(threshold, color="0.5", ls="--", lw=1, label=f"{threshold:g} % of final")
    ax.set_ylabel("hypervolume (% of final)")
    ax.legend()
    ax = axes[1]
    ax.plot(conv["trials"], conv["best_f1"], "o-", color="C2", label="best macro-F1 so far")
    ax.plot(conv["trials"], conv["selected_f1"], "s--", color="C3", label="selected point")
    ax.set_ylabel("macro-F1 (%, validation)")
    ax.legend()
    ax = axes[2]
    ax.plot(conv["trials"], conv["fastest_ms"], "o-", color="C1", label="fastest so far")
    ax.plot(conv["trials"], conv["selected_ms"], "s--", color="C3", label="selected point")
    ax.set_ylabel("testing time per sample (ms, validation)")
    ax.legend()
    for ax in axes:
        ax.set_xlabel("number of trials")
        ax.grid(alpha=0.3)
    fig.suptitle(f"Convergence of the search: {title}")
    fig.tight_layout()
    plt.show()


def min_trials(conv: pd.DataFrame, threshold: float = 99.0) -> int:
    """First checkpoint whose front reaches `threshold` % of the final hypervolume."""
    return int(conv.loc[conv["hypervolume_pct"] >= threshold, "trials"].iloc[0])


def moo_tune(df, dataset_name, label_col="Label", partition_mode="iid", n_clients=10, dirichlet_alpha=0.5,
             n_trials=30, population_size=10, builder_rule="per_class", space=None, f1_tolerance=1.0, n_timing_runs=5,
             val_size=0.20, test_size=0.20, random_state=42, verbose=True, plot=True,
             convergence_every=5) -> Dict:
    """NSGA-II search for the hyperparameters that maximise macro-F1 and minimise testing time per sample.

    Both objectives are measured on a validation split of the TRAINING data (same train/test split as run_eflboost),
    so the test set is never seen. The eFL-Boost algorithm itself is unchanged; only its hyperparameters are searched.
    Every trial trains one model; its first 10, 20, ... rounds are also scored, because a model trained for fewer
    rounds is exactly that prefix (the Builder draws of a round do not depend on later rounds). The Pareto front is taken over all scored
    points, and one point is chosen from it: the fastest point within f1_tolerance macro-F1 points of the best,
    or with f1_tolerance=None the knee point.

    Convergence: after every `convergence_every` trials the front of the trials so far is rebuilt and its hypervolume
    recorded (MOO_FRONTS[...]["convergence"], plotted). The first checkpoint at 99 % of the final hypervolume is a
    lower bound for the number of trials needed; run with a generous n_trials to find it.
    """
    space = space or MOO_SPACE
    feature_cols = [c for c in df.columns if c != label_col]
    X = df[feature_cols].values.astype(np.float64)
    y = df[label_col].values
    X_tr, _, y_tr, _ = train_test_split(X, y, test_size=test_size, random_state=random_state, stratify=y)
    X_fit, X_val, y_fit, y_val = train_test_split(X_tr, y_tr, test_size=val_size, random_state=random_state,
                                                  stratify=y_tr)
    if partition_mode == "iid":
        client_data = partition_iid(X_fit, y_fit, n_clients, seed=random_state)
    else:
        client_data = partition_dirichlet(X_fit, y_fit, n_clients, alpha=dirichlet_alpha, seed=random_state)
    lo, hi, step = space["n_trees"]
    points = []

    def objective(trial):
        p = {
            "n_trees": trial.suggest_int("n_trees", lo, hi, step=step),
            "max_depth": trial.suggest_int("max_depth", *space["max_depth"]),
            "learning_rate": trial.suggest_float("learning_rate", *space["learning_rate"], log=True),
            "n_bins": trial.suggest_categorical("n_bins", space["n_bins"]),
            "lam": trial.suggest_float("lam", *space["lam"], log=True),
        }
        model = EFLBoost(EFLBoostConfig(random_state=random_state, builder_rule=builder_rule, **p))
        with contextlib.redirect_stdout(io.StringIO()):
            model.fit(client_data)
        for n in sorted(set(range(step, p["n_trees"] + 1, step)) | {p["n_trees"]}):
            sub = copy.copy(model)
            sub.trees = model.trees[:n]
            points.append({"trial": trial.number, **p, "n_trees": n,
                           "val_f1_macro": f1_score(y_val, sub.predict(X_val), average="macro", zero_division=0) * 100,
                           "val_time_per_sample_ms": measure_inference_time(sub.predict, X_val, n_timing_runs)[
                               "test_time_per_sample_ms"]})
        if verbose:
            print(f"  trial {trial.number + 1}/{n_trials}: {p} -> F1-macro {points[-1]['val_f1_macro']:.2f} %, "
                  f"{points[-1]['val_time_per_sample_ms']:.4f} ms / sample")
        return points[-1]["val_f1_macro"], points[-1]["val_time_per_sample_ms"]

    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(directions=["maximize", "minimize"],
                                sampler=optuna.samplers.NSGAIISampler(population_size=population_size,
                                                                      seed=random_state))
    t0 = time.perf_counter()
    study.optimize(objective, n_trials=n_trials)

    table = pd.DataFrame(points)
    front = pareto_front(table)
    chosen, rule = select_point(front, f1_tolerance)
    conv = convergence_table(table, f1_tolerance, every=convergence_every)
    casts = {"n_trees": int, "max_depth": int, "learning_rate": float, "n_bins": int, "lam": float}
    params = {k: cast(chosen[k]) for k, cast in casts.items()}
    MOO_FRONTS[(dataset_name, partition_mode)] = {"points": table, "front": front, "chosen": chosen,
                                                  "convergence": conv}

    if verbose:
        print(f"\nMulti-objective search ({dataset_name}, {partition_mode}): {n_trials} trials, {len(table)} points "
              f"scored, {len(front)} on the Pareto front, {time.perf_counter() - t0:.0f} s")
        print(front.round(4).to_string(index=False))
        print(f"\nSelected ({rule}): {params}  ->  validation F1-macro {chosen['val_f1_macro']:.2f} %, "
              f"{chosen['val_time_per_sample_ms']:.4f} ms / sample")
        print(f"\nConvergence (front after every {convergence_every} trials):")
        print(conv.round(4).to_string(index=False))
        print(f"The front reached 99 % of its final hypervolume after {min_trials(conv)} of {n_trials} trials.")
    if plot:
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.scatter(table["val_time_per_sample_ms"], table["val_f1_macro"], s=12, color="0.75", label="evaluated")
        ax.plot(front["val_time_per_sample_ms"], front["val_f1_macro"], "o-", color="C0", label="Pareto front")
        ax.scatter([chosen["val_time_per_sample_ms"]], [chosen["val_f1_macro"]], s=150, marker="*", color="C3",
                   zorder=3, label=f"selected ({rule})")
        ax.set_xlabel("testing time per sample (ms, validation)")
        ax.set_ylabel("macro-F1 (%, validation)")
        ax.set_title(f"{dataset_name} | {partition_mode}")
        ax.legend()
        plt.show()
        plot_convergence(conv, f"{dataset_name} | {partition_mode}")
    return {**params, "builder_rule": builder_rule, "tuned": "moo"}

# Experiments

# 1. CICIDS2017

In [ ]:
df = pd.read_csv("CICIDS2017_sample_km.csv")

# IID

In [ ]:
# ----- IID -----
results_cic_iid = run_eflboost(
    df,
    dataset_name="CICIDS2017",
    label_col="Label",
    partition_mode="iid",
    n_clients=10,
    n_trees=50,
    max_depth=6,
    n_bins=64,
    learning_rate=0.1,
)

### Multi-objective tuning (maximise macro-F1, minimise testing time)

In [ ]:
params_cic_iid = moo_tune(df, "CICIDS2017", partition_mode="iid")

### Train and test with the selected parameters

In [ ]:
results_cic_iid_moo = run_eflboost(
    df,
    dataset_name="CICIDS2017",
    label_col="Label",
    partition_mode="iid",
    n_clients=10,
    **params_cic_iid,
)

# Non-IID

In [ ]:
# ----- Non-IID (Dirichlet) -----
results_cic_noniid = run_eflboost(
    df,
    dataset_name="CICIDS2017",
    label_col="Label",
    partition_mode="dirichlet",
    n_clients=10,
    dirichlet_alpha=0.5,      # smaller = stronger heterogeneity
    n_trees=50,
    max_depth=6,
    n_bins=64,
    learning_rate=0.1,
)

### Multi-objective tuning (maximise macro-F1, minimise testing time)

In [ ]:
params_cic_noniid = moo_tune(df, "CICIDS2017", partition_mode="dirichlet", dirichlet_alpha=0.5)

### Train and test with the selected parameters

In [ ]:
results_cic_noniid_moo = run_eflboost(
    df,
    dataset_name="CICIDS2017",
    label_col="Label",
    partition_mode="dirichlet",
    n_clients=10,
    dirichlet_alpha=0.5,
    **params_cic_noniid,
)

# 2. IoTID2020

In [ ]:
df = pd.read_csv("IoT_Network_Intrusion_Dataset_0.1.csv")

In [ ]:
le = LabelEncoder()
df["Label"] = le.fit_transform(df["Label"])
print("Label mapping:", dict(zip(le.classes_, range(len(le.classes_)))))
# No MinMaxScaler: trees only compare thresholds, so scaling does not change the model.

# IID

In [ ]:
# ----- IID -----
results_iot_iid = run_eflboost(
    df,
    dataset_name="IoTID2020",
    label_col="Label",
    partition_mode="iid",
    n_clients=10,
    n_trees=50,
    max_depth=6,
    n_bins=64,
    learning_rate=0.1,
)

### Multi-objective tuning (maximise macro-F1, minimise testing time)

In [ ]:
params_iot_iid = moo_tune(df, "IoTID2020", partition_mode="iid")

### Train and test with the selected parameters

In [ ]:
results_iot_iid_moo = run_eflboost(
    df,
    dataset_name="IoTID2020",
    label_col="Label",
    partition_mode="iid",
    n_clients=10,
    **params_iot_iid,
)

# Non-IID

In [ ]:
# ----- Non-IID (Dirichlet) -----
results_iot_noniid = run_eflboost(
    df,
    dataset_name="IoTID2020",
    label_col="Label",
    partition_mode="dirichlet",
    n_clients=10,
    dirichlet_alpha=0.5,
    n_trees=50,
    max_depth=6,
    n_bins=64,
    learning_rate=0.1,
)

### Multi-objective tuning (maximise macro-F1, minimise testing time)

In [ ]:
params_iot_noniid = moo_tune(df, "IoTID2020", partition_mode="dirichlet", dirichlet_alpha=0.5)

### Train and test with the selected parameters

In [ ]:
results_iot_noniid_moo = run_eflboost(
    df,
    dataset_name="IoTID2020",
    label_col="Label",
    partition_mode="dirichlet",
    n_clients=10,
    dirichlet_alpha=0.5,
    **params_iot_noniid,
)

# 3. CICEVSE2024

In [ ]:
df = pd.read_csv("CICEVSE2024_12classes_sample.csv")

# IID

In [ ]:
# ----- IID -----
results_evse_iid = run_eflboost(
    df,
    dataset_name="CICEVSE2024",
    label_col="Label",
    partition_mode="iid",
    n_clients=10,
    n_trees=50,
    max_depth=6,
    n_bins=64,
    learning_rate=0.1,
)

### Multi-objective tuning (maximise macro-F1, minimise testing time)

In [ ]:
params_evse_iid = moo_tune(df, "CICEVSE2024", partition_mode="iid")

### Train and test with the selected parameters

In [ ]:
results_evse_iid_moo = run_eflboost(
    df,
    dataset_name="CICEVSE2024",
    label_col="Label",
    partition_mode="iid",
    n_clients=10,
    **params_evse_iid,
)

# Non-IID

In [ ]:
# ----- Non-IID (Dirichlet) -----
results_evse_noniid = run_eflboost(
    df,
    dataset_name="CICEVSE2024",
    label_col="Label",
    partition_mode="dirichlet",
    n_clients=10,
    dirichlet_alpha=0.5,
    n_trees=50,
    max_depth=6,
    n_bins=64,
    learning_rate=0.1,
)

### Multi-objective tuning (maximise macro-F1, minimise testing time)

In [ ]:
params_evse_noniid = moo_tune(df, "CICEVSE2024", partition_mode="dirichlet", dirichlet_alpha=0.5)

### Train and test with the selected parameters

In [ ]:
results_evse_noniid_moo = run_eflboost(
    df,
    dataset_name="CICEVSE2024",
    label_col="Label",
    partition_mode="dirichlet",
    n_clients=10,
    dirichlet_alpha=0.5,
    **params_evse_noniid,
)

# Summary of all runs

In [ ]:
summary = pd.DataFrame(ALL_RESULTS)
summary.to_csv("eflboost_perclass_results.csv", index=False)
cols = ["dataset", "partition", "builder_rule", "tuned", "n_boosting_rounds", "max_depth", "learning_rate", "n_bins", "lam",
        "accuracy", "f1_macro", "f1_weighted", "train_time_s", "test_time_per_sample_ms",
        "model_size_mb", "n_trees", "n_nodes", "n_comm_rounds", "total_comm_mb"]
display(summary.reindex(columns=cols).round(4))

# Comparison with the round-robin Builder (eFLBoost_1.ipynb)

`eFLBoost_1.ipynb` saves its runs to `eflboost_results.csv`. Put that file next to this notebook to compare the two
Builder rules run for run (same datasets, partitions, split and search).

In [ ]:
import os
if os.path.exists("eflboost_results.csv"):
    rr = pd.read_csv("eflboost_results.csv").assign(builder_rule="round_robin")
    both = pd.concat([rr, pd.DataFrame(ALL_RESULTS)], ignore_index=True)
    both["tuned"] = both["tuned"].astype(str).replace({"False": "original params", "moo": "MOO"})
    table = both.pivot_table(index=["dataset", "partition", "tuned"], columns="builder_rule",
                             values=["f1_macro", "test_time_per_sample_ms", "total_comm_mb", "n_comm_rounds"])
    display(table.round(4))
else:
    print("eflboost_results.csv (from eFLBoost_1.ipynb) not found: run that notebook's summary cell first.")